# Baselines y seis candidatos: comparación por el criterio del negocio

Ejecutar con Restart / Run All después de completar las fases `modelos` y `ablacion` del coordinador. Las salidas guardadas y el manifiesto de verificación acreditan la ejecución; una plantilla sin salidas sigue pendiente. Se importan el Pipeline y los módulos versionados; las búsquedas ya registradas no se repiten desde el notebook.

La selección prioriza la media entre folds de cobertura de Fail con cupo del 10 % por edición. AP es la métrica técnica común; Brier y log-loss diagnostican calibración. La desviación entre folds es dispersión descriptiva, no un intervalo de confianza.


In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset-03/v2/config/protocolo-v2.json').is_file():
    ROOT = here/'pset-03/v2'
assert ROOT is not None, 'Abrir el notebook desde v2/notebooks o desde la raíz del repositorio'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


Protocolo verificado: 07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b24172c56aaf18e8ab0d
Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.


In [2]:
models = report('docs/modelos/resultados.json')
assert models['status'] == 'complete' and models['protocol_id'] == plan['id']
assert len(models['baselines']) == 2 and len(models['candidates']) == 6
from src.modelos import candidatos
rows = []
for result in models['candidates']:
    assert len(result['folds']) == 5
    np.testing.assert_allclose(result['cv_coverage_mean'], np.mean([f['cv_coverage'] for f in result['folds']]))
    estimator = candidatos()[result['family']]
    params = estimator.get_params(deep=False)
    rows.append({'familia':result['family'], 'AP_train_muestra':result['train_ap_fail_mean'],
                 'AP_train_DE':float(np.std([f['train_ap_fail'] for f in result['folds']], ddof=1)),
                 'AP_CV_media':result['cv_ap_fail_mean'], 'AP_CV_DE':result['cv_ap_fail_std'],
                 'brecha_AP':result['ap_gap'], 'cobertura_CV_media':result['cv_coverage_mean'],
                 'cobertura_CV_DE':result['cv_coverage_std'], 'Brier_CV':result['cv_brier_fail_mean'],
                 'logloss_CV':result['cv_logloss_mean'], 'fit_CV_seg':result['cv_fit_seconds_total'],
                 'inferencia_CV_seg':result['cv_inference_seconds_total'],
                 'API_costo_nuevo_USD':result.get('api_cost_new_calls_usd',0),
                 'learning_rate':params.get('learning_rate_init',params.get('learning_rate','no aplica')),
                 'run_id':result['run_id']})
comparison = pd.DataFrame(rows).sort_values('cobertura_CV_media',ascending=False)
display(comparison)
display(pd.DataFrame([{'baseline':'Dummy más frecuente', 'cobertura_CV':models['baselines'][0]['cv_coverage_mean'], 'AP_CV':models['baselines'][0]['cv_ap_fail_mean']},
                      {'baseline':'Reactiva al cierre', 'cobertura_CV':models['baselines'][1]['preventive_coverage'], 'AP_CV':np.nan}]))


,familia,AP_train_muestra,AP_train_DE,AP_CV_media,AP_CV_DE,brecha_AP,cobertura_CV_media,cobertura_CV_DE,Brier_CV,logloss_CV,fit_CV_seg,inferencia_CV_seg,API_costo_nuevo_USD,learning_rate,run_id
0,lineal,0.584077,0.027741,0.568740,0.024483,0.015337,0.229226,0.008012,0.174711,0.830963,2.757541,0.432196,0.000000,no aplica,7f75bbbab6ed4f56b58caa0850529b34
2,boosting,0.726866,0.039078,0.577654,0.027252,0.149212,0.228074,0.008209,0.173761,0.818299,5.160584,0.217157,0.000000,0.1,61ef18564a504ec4bc39e454cbe1f8e9
4,bayes,0.543219,0.023436,0.537542,0.027301,0.005677,0.227818,0.010680,0.238012,2.740578,0.876588,0.183671,0.000000,no aplica,dccc18435ccf489794cadd3ce4903342
1,bagging,1.000000,0.000000,0.561059,0.030092,0.438941,0.225686,0.011184,0.176665,0.838080,15.632964,0.526103,0.000000,no aplica,5a936638bef24491932483e0c0ede0df
3,red_neuronal,0.777387,0.026289,0.504241,0.028346,0.273146,0.200911,0.012721,0.201395,0.987472,328.081107,0.221983,0.000000,0.001,7ce1fba214954655bf54f7d8e9a83842
5,llm_generativo,0.392501,0.038523,0.382111,0.039352,0.010389,0.148656,0.018812,0.215397,1.001499,1.157747,349.805297,0.620597,no aplica,f4e06d2464e44b1da84b5c7f2ecf55b5


,baseline,cobertura_CV,AP_CV
0,Dummy más frecuente,0.091542,0.30115
1,Reactiva al cierre,0.000000,NaN


## Cómo interpretar entrenamiento, recursos y sobreajuste

La métrica de entrenamiento usa exactamente la misma muestra estratificada de hasta 1.000 filas por train-fold para los seis candidatos. Se identifica como muestra y no como la métrica de todo el entrenamiento. La validación incluye todas las mismas matrículas y folds para los seis.

El GPT usa 18 ejemplos de train-fold y sus proporciones de clase; no ajusta sus pesos con todas las filas. Su fit construye el contexto y su mayor costo ocurre al predecir. Se reportan inferencia, latencia y consumo además del tiempo de fit. El límite global incluye pilotos e intentos interrumpidos; el costo nuevo del run no equivale al consumo total del proyecto. La reproducción exacta usa respuestas archivadas y no presupone determinismo del proveedor.


In [3]:
for row in models['candidates']:
    warnings = [w for fold in row['folds'] for w in fold.get('warnings',[])]
    print(row['family'], '| brecha AP:',round(row['ap_gap'],4), '| avisos registrados:',len(warnings))
    if warnings:
        display(pd.DataFrame(warnings).drop_duplicates().head(5))


lineal | brecha AP: 0.0153 | avisos registrados: 0
bagging | brecha AP: 0.4389 | avisos registrados: 0
boosting | brecha AP: 0.1492 | avisos registrados: 0
red_neuronal | brecha AP: 0.2731 | avisos registrados: 5


,category,message
0,ConvergenceWarning,Stochastic Optimizer: Maximum iterations (200)...


bayes | brecha AP: 0.0057 | avisos registrados: 0
llm_generativo | brecha AP: 0.0104 | avisos registrados: 0


## Aporte de las variables derivadas

La ablación compara una arquitectura lineal fija y los mismos cinco folds: estáticas, longitudinal original, solo log1p, solo intensidad normalizada y ambas derivadas. Se conservan los resultados negativos. Una mejora menor que la variación entre folds no se presenta como una mejora demostrada.


In [4]:
ablation = report('docs/ablacion/resultados.json')
assert ablation['protocol_id'] == plan['id'] and ablation['test_used'] is False
display(pd.DataFrame([{k:v for k,v in r.items() if k!='folds'} for r in ablation['results']]))


,mode,protocol_id,run_id,ap_fail_mean,ap_fail_std,coverage_mean,coverage_std,train_ap_fail_mean,train_ap_fail_std,fit_seconds_mean,fit_seconds_std,delta_ap_vs_original,delta_coverage_vs_original,coverage_gain_exceeds_fold_variation
0,static,07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b2...,ff7db109d35243509fac1035ea3173a9,0.423200,0.018213,0.160451,0.003524,0.440625,0.022435,0.446181,0.175317,-0.145789,-0.067737,False
1,original,07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b2...,a1530c05f1af40e6a55db02a47c86edd,0.568988,0.024948,0.228188,0.007872,0.584246,0.028027,0.566920,0.161619,0.000000,0.000000,False
2,log_only,07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b2...,9bd87641c70443229b47e03103502d3c,0.568978,0.024478,0.229215,0.007891,0.584201,0.027415,0.699590,0.267415,-0.000010,0.001026,False
3,intensity_only,07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b2...,356436c87730462a86b638b8891f774b,0.568782,0.024608,0.229430,0.008244,0.584319,0.028151,0.512607,0.056338,-0.000206,0.001242,False
4,both,07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b2...,895aad897f124389a37197b76ad280cb,0.568740,0.024483,0.229226,0.008012,0.584077,0.027741,0.493349,0.053387,-0.000248,0.001037,False
